In [1]:
!pip install -q transformers datasets sentencepiece accelerate evaluate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.6 MB/s eta 0:00:00


In [2]:
import torch
import transformers
import datasets

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("Datasets version:", datasets.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.10.0+cpu
Transformers version: 5.0.0
Datasets version: 4.0.0
GPU available: False


In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu128
GPU available: True
GPU: Tesla T4


In [2]:
from datasets import load_dataset

dataset = load_dataset("cfilt/iitb-english-hindi")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/500k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1659083
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 520
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2507
    })
})


In [3]:
print(dataset["train"][0])

{'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}}


In [4]:
print("English:", dataset["train"][0]["translation"]["en"])
print("Hindi:", dataset["train"][0]["translation"]["hi"])

English: Give your application an accessibility workout
Hindi: अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें


In [5]:
print("Training data:", len(dataset["train"]))
print("Validation data:", len(dataset["validation"]))
print("Test data:", len(dataset["test"]))

Training data: 1659083
Validation data: 520
Test data: 2507


In [6]:
train_data = dataset["train"].select(range(10000))

print("Training examples:", len(train_data))

Training examples: 10000


In [8]:
val_data = dataset["validation"].select(range(520))
test_data = dataset["test"].select(range(1000))

print("Validation examples:", len(val_data))
print("Test examples:", len(test_data))

Validation examples: 520
Test examples: 1000


In [9]:
print(train_data[0])
print(val_data[0])
print(test_data[0])

{'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}}
{'translation': {'en': 'Students of the Dattatreya city Municipal corporation secondary school demonstrated their imagination power by creating the fictitious fort "Duttgarh".', 'hi': "महानगर पालिका अंतर्गत दत्तात्रय नगर माध्यमिक स्कूल के विद्यार्थियों ने काल्पनिक किला 'दत्तगढ़' बनाकर अपनी कल्पनाशक्ति का परिचय दिया।"}}
{'translation': {'en': 'A black box in your car?', 'hi': 'आपकी कार में ब्लैक बॉक्स?'}}


In [10]:
from transformers import AutoTokenizer

model_name = "Helsinki-NLP/opus-mt-en-hi"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully!


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [11]:
text = "How are you?"

tokens = tokenizer(text)

print(tokens)

{'input_ids': [244, 54, 27, 22, 0], 'attention_mask': [1, 1, 1, 1, 1]}


In [12]:
hindi_text = "आप कैसे हैं?"

tokens = tokenizer(hindi_text)

print(tokens)

{'input_ids': [44, 3605, 2703, 44, 22216, 44, 6286, 549, 22, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [13]:
def preprocess_function(examples):
    inputs = [item["en"] for item in examples["translation"]]
    targets = [item["hi"] for item in examples["translation"]]

    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [14]:
tokenized_train = train_data.map(
    preprocess_function,
    batched=True
)

print(tokenized_train)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset({
    features: ['translation', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 10000
})


In [15]:
tokenized_val = val_data.map(
    preprocess_function,
    batched=True
)

tokenized_test = test_data.map(
    preprocess_function,
    batched=True
)

print("Training:", len(tokenized_train))
print("Validation:", len(tokenized_val))
print("Test:", len(tokenized_test))

Map:   0%|          | 0/520 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Training: 10000
Validation: 520
Test: 1000


In [16]:
print(tokenized_train[0])

{'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}, 'input_ids': [3872, 85, 2501, 132, 15441, 36398, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1], 'labels': [63, 2025, 18, 16155, 346, 20311, 24, 2279, 679, 0]}


In [17]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Transformer model loaded successfully!")

pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Transformer model loaded successfully!


In [18]:
print(type(model))

<class 'transformers.models.marian.modeling_marian.MarianMTModel'>


In [19]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

print("Data collator created successfully!")

Data collator created successfully!


In [20]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [21]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./english_hindi_model",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    report_to="none"
)

print("Training configuration created!")

Training configuration created!


In [22]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

print("Trainer created successfully!")

Trainer created successfully!


In [24]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.398839,4.502147


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.43009022064208985, metrics={'train_runtime': 118.601, 'train_samples_per_second': 84.316, 'train_steps_per_second': 10.54, 'total_flos': 29843273023488.0, 'train_loss': 0.43009022064208985, 'epoch': 1.0})

In [25]:
results = trainer.evaluate()

print(results)

{'eval_loss': 4.502147197723389, 'eval_runtime': 2.2918, 'eval_samples_per_second': 226.901, 'eval_steps_per_second': 28.363, 'epoch': 1.0}


In [26]:
text = "The patient has a fever."

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_length=128
)

translation = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("English:", text)
print("Hindi:", translation)

English: The patient has a fever.
Hindi: मरीज़ को बुखार है.


In [27]:
sentences = [
    "The patient has a fever.",
    "The doctor examined the patient.",
    "The computer is connected to the network.",
    "The sample was collected from the laboratory."
]

for text in sentences:
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=128)

    translation = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("English:", text)
    print("Hindi:", translation)
    print("-" * 50)

English: The patient has a fever.
Hindi: मरीज़ को बुखार है.
--------------------------------------------------
English: The doctor examined the patient.
Hindi: डॉक्टर ने मरीज़ की जाँच की ।
--------------------------------------------------
English: The computer is connected to the network.
Hindi: कम्प्यूटर नेटवर्क से जुड़ा हुआ है.
--------------------------------------------------
English: The sample was collected from the laboratory.
Hindi: नमूना प्रयोगशाला से इकट्ठा किया गया था.
--------------------------------------------------
